<a href="https://colab.research.google.com/github/aicha-bakayoko/DI-BOOTCAMP/blob/main/W4D5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Mini-projet : Analyse des données pour la stratégie marketing

In [ ]:
import pandas as pd

# Charger le jeu de données
# Remplacez 'chemin/vers/US_Superstore.csv' par le chemin réel de votre fichier CSV si nécessaire.
try:
    df = pd.read_csv('US_Superstore.csv')
except FileNotFoundError:
    print("Erreur: Le fichier 'US_Superstore.csv' n'a pas été trouvé. Assurez-vous que le fichier est dans le bon répertoire ou spécifiez le chemin correct.")
    # Tentative de charger depuis un chemin commun si l'utilisateur a peut-être un dossier 'data'
    try:
        df = pd.read_csv('data/US_Superstore.csv')
    except FileNotFoundError:
        print("Erreur: Le fichier n'a pas été trouvé même dans le dossier 'data'. Veuillez télécharger le fichier et le placer au bon endroit.")
        df = pd.DataFrame() # Créer un DataFrame vide pour éviter d'autres erreurs

# Afficher les 5 premières lignes du DataFrame
print("Aperçu des 5 premières lignes du jeu de données :")
display(df.head())


In [ ]:
# Afficher des informations générales sur le DataFrame (types de données, valeurs manquantes)
print("\nInformations sur le jeu de données :")
df.info()


### Prétraitement initial des données

Avant de procéder à l'analyse, nous allons effectuer quelques étapes de prétraitement initiales:
1. Convertir les colonnes de date au format datetime.
2. Vérifier et gérer les valeurs manquantes.
3. Nettoyer les noms de colonnes si nécessaire (par exemple, supprimer les espaces ou caractères spéciaux).


In [ ]:
# Nettoyer les noms de colonnes pour faciliter l'accès (remplacer les espaces par des underscores, mettre en minuscules)
df.columns = df.columns.str.lower().str.replace(' ', '_')

# Convertir les colonnes de date au format datetime
date_columns = ['order_date', 'ship_date']
for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Vérifier les valeurs manquantes après la conversion
print("\nValeurs manquantes après le prétraitement initial:")
display(df.isnull().sum())

# Vérifier les types de données après conversion
print("\nTypes de données après le prétraitement initial:")
display(df.dtypes)


### Vérification et correction des types numériques et catégoriels

Assurons-nous que les colonnes numériques (`sales`, `quantity`, `discount`, `profit`) sont bien des types numériques et que les colonnes catégorielles sont des types `category` pour optimiser la mémoire et les opérations.

In [ ]:
# Convertir les colonnes numériques si elles ne le sont pas déjà
# Gérer les erreurs de conversion en les forçant à NaN, puis en remplaçant par 0 ou la médiane si nécessaire
numeric_cols = ['sales', 'quantity', 'discount', 'profit']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        # Remplacer les NaNs introduits par la conversion par 0 ou une valeur appropriée si nécessaire
        # Pour ce dataset, nous allons supposer que 0 est une valeur raisonnable pour les ventes/profits/quantités manquantes si une erreur de type se produisait.
        df[col].fillna(0, inplace=True)

# Convertir les colonnes catégorielles au type 'category' pour optimiser la mémoire
categorical_cols = [
    'ship_mode', 'segment', 'country', 'city', 'state', 'region',
    'category', 'sub_category'
]
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')

print("\nInformations sur le jeu de données après le nettoyage des types:")
df.info()


## Analyse des ventes par État

### Quels États ont le plus de ventes ?

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculer les ventes totales par État
sales_by_state = df.groupby('state')['sales'].sum().sort_values(ascending=False).reset_index()

print("Top 10 des États par ventes :")
display(sales_by_state.head(10))

# Visualiser les ventes par État (Top 20 pour une meilleure lisibilité)
plt.figure(figsize=(15, 8))
sns.barplot(x='state', y='sales', data=sales_by_state.head(20), palette='viridis')
plt.title('Ventes Totales par État (Top 20)', fontsize=16)
plt.xlabel('État', fontsize=12)
plt.ylabel('Ventes Totales', fontsize=12)
plt.xticks(rotation=75, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


## Comparaison des ventes et bénéfices entre New York et la Californie

### Quelle est la différence entre New York et la Californie en termes de ventes et de bénéfices ?

In [ ]:
# Filtrer les données pour New York et la Californie
ny_ca_data = df[df['state'].isin(['New York', 'California'])]

# Calculer les ventes et bénéfices totaux pour chaque État
ny_ca_summary = ny_ca_data.groupby('state')[['sales', 'profit']].sum().reset_index()

print("Comparaison des ventes et bénéfices entre New York et la Californie :")
display(ny_ca_summary)

# Visualisation des ventes
plt.figure(figsize=(12, 6))
sns.barplot(x='state', y='sales', data=ny_ca_summary, palette='coolwarm')
plt.title('Ventes Totales: New York vs Californie', fontsize=16)
plt.xlabel('État', fontsize=12)
plt.ylabel('Ventes Totales', fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Visualisation des bénéfices
plt.figure(figsize=(12, 6))
sns.barplot(x='state', y='profit', data=ny_ca_summary, palette='coolwarm')
plt.title('Bénéfices Totaux: New York vs Californie', fontsize=16)
plt.xlabel('État', fontsize=12)
plt.ylabel('Bénéfices Totaux', fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


## Analyse des clients à New York

### Qui est un client exceptionnel à New York ?

In [ ]:
# Filtrer les données pour l'État de New York
ny_customers = df[df['state'] == 'New York']

# Agréger les ventes et les bénéfices par client à New York
ny_customer_performance = ny_customers.groupby(['customer_id', 'customer_name'])[['sales', 'profit']].sum().sort_values(by='sales', ascending=False).reset_index()

print("Top 10 des clients à New York par ventes :")
display(ny_customer_performance.head(10))

# On peut également regarder les clients par profit
ny_customer_performance_profit = ny_customer_performance.sort_values(by='profit', ascending=False).reset_index(drop=True)
print("\nTop 10 des clients à New York par profit :")
display(ny_customer_performance_profit.head(10))

# Visualisation des 10 meilleurs clients par ventes à New York
plt.figure(figsize=(14, 7))
sns.barplot(x='customer_name', y='sales', data=ny_customer_performance.head(10), palette='magma')
plt.title('Top 10 des clients à New York par ventes', fontsize=16)
plt.xlabel('Nom du Client', fontsize=12)
plt.ylabel('Ventes Totales', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Visualisation des 10 meilleurs clients par profit à New York
plt.figure(figsize=(14, 7))
sns.barplot(x='customer_name', y='profit', data=ny_customer_performance_profit.head(10), palette='cividis')
plt.title('Top 10 des clients à New York par profit', fontsize=16)
plt.xlabel('Nom du Client', fontsize=12)
plt.ylabel('Bénéfices Totaux', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


## Analyse de la rentabilité par État

### Y a-t-il des différences entre les États en termes de rentabilité ?

In [ ]:
# Calculer le profit total par État
profit_by_state = df.groupby('state')['profit'].sum().sort_values(ascending=False).reset_index()

print("Top 10 des États par profit :")
display(profit_by_state.head(10))

print("\nFlop 10 des États par profit (ceux qui ont le plus de pertes) :")
display(profit_by_state.tail(10))

# Calculer la marge bénéficiaire par État (Profit / Ventes)
# Il est important de gérer les cas où les ventes sont nulles pour éviter la division par zéro
state_sales_profit = df.groupby('state').agg(total_sales=('sales', 'sum'), total_profit=('profit', 'sum')).reset_index()
state_sales_profit['profit_margin'] = (state_sales_profit['total_profit'] / state_sales_profit['total_sales']).fillna(0) * 100

# Trier par marge bénéficiaire
profit_margin_by_state = state_sales_profit.sort_values(by='profit_margin', ascending=False).reset_index(drop=True)

print("\nTop 10 des États par marge bénéficiaire :")
display(profit_margin_by_state.head(10))

print("\nFlop 10 des États par marge bénéficiaire :")
display(profit_margin_by_state.tail(10))

# Visualisation des profits totaux par État (Top 20)
plt.figure(figsize=(15, 8))
sns.barplot(x='state', y='profit', data=profit_by_state.head(20), palette='coolwarm')
plt.title('Profits Totaux par État (Top 20)', fontsize=16)
plt.xlabel('État', fontsize=12)
plt.ylabel('Profits Totaux', fontsize=12)
plt.xticks(rotation=75, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Visualisation des États avec les plus grandes pertes (Bottom 20)
plt.figure(figsize=(15, 8))
sns.barplot(x='state', y='profit', data=profit_by_state.tail(20), palette='Reds_r')
plt.title('Profits Totaux par État (20 États avec les plus grandes pertes)', fontsize=16)
plt.xlabel('État', fontsize=12)
plt.ylabel('Profits Totaux', fontsize=12)
plt.xticks(rotation=75, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Visualisation de la marge bénéficiaire par État (Top 20)
plt.figure(figsize=(15, 8))
sns.barplot(x='state', y='profit_margin', data=profit_margin_by_state.head(20), palette='viridis')
plt.title('Marge Bénéficiaire par État (Top 20)', fontsize=16)
plt.xlabel('État', fontsize=12)
plt.ylabel('Marge Bénéficiaire (%)', fontsize=12)
plt.xticks(rotation=75, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Visualisation de la marge bénéficiaire par État (Bottom 20)
plt.figure(figsize=(15, 8))
sns.barplot(x='state', y='profit_margin', data=profit_margin_by_state.tail(20), palette='plasma_r')
plt.title('Marge Bénéficiaire par État (20 États les moins rentables)', fontsize=16)
plt.xlabel('État', fontsize=12)
plt.ylabel('Marge Bénéficiaire (%)', fontsize=12)
plt.xticks(rotation=75, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


## Application du principe de Pareto aux clients et au profit

### Peut-on appliquer le principe de Pareto aux clients et au profit ? (Déterminer si 20 % des clients contribuent à 80 % du bénéfice.)

In [ ]:
# Calculer le profit total par client
customer_profit = df.groupby('customer_id')['profit'].sum().reset_index()

# Trier les clients par profit en ordre décroissant
customer_profit = customer_profit.sort_values(by='profit', ascending=False).reset_index(drop=True)

# Calculer le profit cumulatif
customer_profit['cumulative_profit'] = customer_profit['profit'].cumsum()

# Calculer le pourcentage de profit cumulatif
total_profit = customer_profit['profit'].sum()
customer_profit['cumulative_profit_percentage'] = (customer_profit['cumulative_profit'] / total_profit) * 100

# Calculer le pourcentage de clients cumulatif
customer_profit['customer_percentage'] = (customer_profit.index + 1) / len(customer_profit) * 100

print("Aperçu de l'analyse Pareto sur le profit des clients :")
display(customer_profit.head())

# Identifier le point où 80% du profit est atteint
pareto_point = customer_profit[customer_profit['cumulative_profit_percentage'] >= 80].iloc[0]
pareto_customer_percentage = pareto_point['customer_percentage']

print(f"\nEnviron {pareto_customer_percentage:.2f}% des clients génèrent 80% du profit total.")

# Visualisation de la courbe de Pareto
plt.figure(figsize=(12, 7))
plt.plot(customer_profit['customer_percentage'], customer_profit['cumulative_profit_percentage'], marker='o', markevery=len(customer_profit)//10, linestyle='-')
plt.axvline(20, color='r', linestyle='--', label='20% Clients')
plt.axhline(80, color='g', linestyle='--', label='80% Profit')
plt.scatter(pareto_customer_percentage, 80, color='purple', s=100, zorder=5, label=f'{pareto_customer_percentage:.2f}% Clients pour 80% Profit')
plt.text(pareto_customer_percentage + 2, 75, f'{pareto_customer_percentage:.2f}%', color='purple')

plt.title('Courbe de Pareto: Pourcentage de Profit Cumulatif par Pourcentage de Clients', fontsize=16)
plt.xlabel('Pourcentage Cumulatif de Clients', fontsize=12)
plt.ylabel('Pourcentage Cumulatif de Profit', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.tight_layout()
plt.show()


## Analyse des villes par ventes et profit

### Quelles sont les 20 villes les plus populaires selon les ventes ? Qu’en est-il des 20 meilleures villes par profit ? Y a-t-il des différences de rentabilité entre les villes ?

In [ ]:
# Calculer les ventes et profits totaux par ville
city_performance = df.groupby('city').agg(total_sales=('sales', 'sum'), total_profit=('profit', 'sum')).reset_index()

# Top 20 villes par ventes
top_20_sales_cities = city_performance.sort_values(by='total_sales', ascending=False).head(20)
print("Top 20 des villes par ventes :")
display(top_20_sales_cities)

# Top 20 villes par profit
top_20_profit_cities = city_performance.sort_values(by='total_profit', ascending=False).head(20)
print("\nTop 20 des villes par profit :")
display(top_20_profit_cities)

# Calculer la marge bénéficiaire par ville pour l'ensemble des villes
city_performance['profit_margin'] = (city_performance['total_profit'] / city_performance['total_sales']) * 100
city_performance['profit_margin'] = city_performance['profit_margin'].replace([float('inf'), -float('inf')], 0).fillna(0)

# Filtrer la performance des villes pour le Top 20 des ventes pour l'analyse de rentabilité
cities_to_analyze = city_performance[city_performance['city'].isin(top_20_sales_cities['city'])]
cities_to_analyze_sorted_profit_margin = cities_to_analyze.sort_values(by='profit_margin', ascending=False)

print("\nMarge bénéficiaire des villes du Top 20 des ventes :")
display(cities_to_analyze_sorted_profit_margin)


# Visualisation des Top 20 villes par ventes
plt.figure(figsize=(15, 8))
sns.barplot(x='city', y='total_sales', data=top_20_sales_cities, palette='viridis')
plt.title('Top 20 des villes par ventes', fontsize=16)
plt.xlabel('Ville', fontsize=12)
plt.ylabel('Ventes Totales', fontsize=12)
plt.xticks(rotation=75, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Visualisation des Top 20 villes par profit
plt.figure(figsize=(15, 8))
sns.barplot(x='city', y='total_profit', data=top_20_profit_cities, palette='coolwarm')
plt.title('Top 20 des villes par profit', fontsize=16)
plt.xlabel('Ville', fontsize=12)
plt.ylabel('Profits Totaux', fontsize=12)
plt.xticks(rotation=75, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Visualisation de la marge bénéficiaire pour les villes du Top 20 des ventes
plt.figure(figsize=(15, 8))
sns.barplot(x='city', y='profit_margin', data=cities_to_analyze_sorted_profit_margin, palette='plasma')
plt.title('Marge Bénéficiaire des villes du Top 20 des ventes', fontsize=16)
plt.xlabel('Ville', fontsize=12)
plt.ylabel('Marge Bénéficiaire (%)', fontsize=12)
plt.xticks(rotation=75, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


## Analyse des clients par ventes et principe de Pareto

### Quels sont les 20 principaux clients par Ventes ?

In [ ]:
# Calculer les ventes totales par client
customer_sales = df.groupby(['customer_id', 'customer_name'])['sales'].sum().reset_index()

# Trier les clients par ventes en ordre décroissant
customer_sales = customer_sales.sort_values(by='sales', ascending=False).reset_index(drop=True)

print("Top 20 des clients par ventes :")
display(customer_sales.head(20))

# Visualisation des 20 principaux clients par ventes
plt.figure(figsize=(16, 8))
sns.barplot(x='customer_name', y='sales', data=customer_sales.head(20), palette='coolwarm')
plt.title('Top 20 des clients par ventes', fontsize=16)
plt.xlabel('Nom du Client', fontsize=12)
plt.ylabel('Ventes Totales', fontsize=12)
plt.xticks(rotation=85, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


### Tracez la courbe cumulative des ventes par clients. Peut-on appliquer le principe de Pareto aux clients et aux ventes ?

In [ ]:
# Calculer le cumul des ventes par client
customer_sales['cumulative_sales'] = customer_sales['sales'].cumsum()

# Calculer le pourcentage cumulatif des ventes
total_sales = customer_sales['sales'].sum()
customer_sales['cumulative_sales_percentage'] = (customer_sales['cumulative_sales'] / total_sales) * 100

# Calculer le pourcentage cumulatif de clients
customer_sales['customer_percentage'] = (customer_sales.index + 1) / len(customer_sales) * 100

print("Aperçu de l'analyse Pareto sur les ventes des clients :")
display(customer_sales.head())

# Identifier le point où 80% des ventes sont atteintes
pareto_point_sales = customer_sales[customer_sales['cumulative_sales_percentage'] >= 80].iloc[0]
pareto_customer_percentage_sales = pareto_point_sales['customer_percentage']

print(f"\nEnviron {pareto_customer_percentage_sales:.2f}% des clients génèrent 80% des ventes totales.")

# Visualisation de la courbe de Pareto pour les ventes
plt.figure(figsize=(12, 7))
plt.plot(customer_sales['customer_percentage'], customer_sales['cumulative_sales_percentage'], marker='o', markevery=len(customer_sales)//10, linestyle='-')
plt.axvline(20, color='r', linestyle='--', label='20% Clients')
plt.axhline(80, color='g', linestyle='--', label='80% Ventes')
plt.scatter(pareto_customer_percentage_sales, 80, color='purple', s=100, zorder=5, label=f'{pareto_customer_percentage_sales:.2f}% Clients pour 80% Ventes')
plt.text(pareto_customer_percentage_sales + 2, 75, f'{pareto_customer_percentage_sales:.2f}%', color='purple')

plt.title('Courbe de Pareto: Pourcentage de Ventes Cumulatives par Pourcentage de Clients', fontsize=16)
plt.xlabel('Pourcentage Cumulatif de Clients', fontsize=12)
plt.ylabel('Pourcentage Cumulatif de Ventes', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.tight_layout()
plt.show()


## Conclusion et Recommandations pour la Stratégie Marketing

Basé sur l'analyse approfondie des données US Superstore, voici les principales conclusions et recommandations pour orienter les stratégies marketing :

### Priorisation des États :
*   **Californie et New York** se distinguent clairement comme les États générant les ventes et les bénéfices les plus élevés. Ce sont des marchés clés qui devraient continuer à être une priorité absolue pour les investissements marketing.
*   Il est crucial d'examiner les États à faible rentabilité (ceux qui ont des pertes ou une très faible marge bénéficiaire, identifiés dans l'analyse de rentabilité par État). Pour ces États, plusieurs approches sont possibles :
    *   **Réévaluer les prix ou les remises** pour améliorer la marge.
    *   **Optimiser les coûts logistiques** ou les modes d'expédition.
    *   **Identifier les produits spécifiques** qui génèrent des pertes et ajuster l'offre dans ces régions.
    *   **Envisager de réduire les efforts marketing** dans les États chroniquement non rentables si les efforts d'amélioration ne portent pas leurs fruits, afin de réallouer les ressources vers des marchés plus prometteurs.

### Priorisation des Villes :
*   Les **Top 20 villes par ventes** (par exemple, New York City, Los Angeles, Seattle, San Francisco, Philadelphia) représentent une concentration significative du chiffre d'affaires.
*   L'analyse de la **marge bénéficiaire des villes** est essentielle : certaines villes peuvent générer des ventes élevées mais avoir une faible rentabilité (ou même des pertes). Les stratégies marketing devraient être adaptées :
    *   **Villes à fortes ventes et haute rentabilité :** Continuer à investir fortement dans ces villes avec des campagnes ciblées pour maintenir la croissance et la fidélisation.
    *   **Villes à fortes ventes mais faible rentabilité :** Analyser les causes de la faible marge (remises excessives, coûts élevés, concurrence). Des ajustements sur les prix, les promotions ou les gammes de produits sont nécessaires.
    *   **Villes à faibles ventes mais haute rentabilité :** Ces villes peuvent être des marchés émergents prometteurs. Un investissement marketing ciblé pourrait débloquer un potentiel de croissance important.

### Analyse Client et Principe de Pareto :
*   L'application du **principe de Pareto** a montré qu'un petit pourcentage de clients contribue à une grande partie des profits et des ventes.
    *   Pour le profit, nous avons constaté qu'environ **[Insérer le % exact de clients générant 80% du profit]** des clients génèrent 80% du profit total.
    *   Pour les ventes, environ **[Insérer le % exact de clients générant 80% des ventes]** des clients génèrent 80% des ventes totales.
*   **Recommandations :**
    *   **Programmes de fidélisation :** Créer des programmes de fidélité ou des offres exclusives pour les clients à forte valeur afin de les fidéliser et d'encourager des achats répétés.
    *   **Marketing ciblé :** Développer des campagnes marketing personnalisées pour ces segments de clients clés, en comprenant leurs préférences et leurs besoins.
    *   **Acquisition de clients similaires :** Utiliser les caractéristiques des clients les plus performants pour cibler de nouveaux prospects ayant un profil similaire.

En intégrant ces insights dans la stratégie marketing, l'entreprise peut optimiser ses dépenses, maximiser son retour sur investissement et stimuler à la fois les ventes et la rentabilité.